In [ ]:
#pip install "gymnasium[box2d]"


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gymnasium as gym

# 1. Create the environment
# render_mode="human" tells Gymnasium to open a window and show the visual
env = gym.make("LunarLander-v3", render_mode="human")

# 2. Reset the environment to start a new episode
observation, info = env.reset(seed=42)

# 3. Run a loop for 1000 frames
for _ in range(1000):
    # Select a random action from the action space
    # 0 = Do nothing, 1 = Fire left engine, 2 = Fire main engine, 3 = Fire right engine
    action = env.action_space.sample()
    
    # Apply the action to the environment
    observation, reward, terminated, truncated, info = env.step(action)
    
    # If the episode finishes (lander crashes or lands safely), reset it
    if terminated or truncated:
        observation, info = env.reset()

# 4. Close the environment when done
env.close()

#env.action_space.sample() — picks a random action (0-3)
# from a uniform distribution. It never looks at observation
# it's just an RNG draw, 
# same odds every time regardless of what the lander is doing.

Trial

In [7]:
# If needed (first time only):
# pip install "gymnasium[box2d]"
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random



# 1. Create the environment
# render_mode="human" tells Gymnasium to open a window and show the visual
env = gym.make("LunarLander-v3", render_mode="human")

# 2. Reset the environment to start a new episode
observation, info = env.reset(seed=42)


# 3. Run a loop for 1000 frames
for _ in range(1000):
    # Select a random action from the action space
    # 0 = Do nothing, 1 = Fire left engine, 2 = Fire main engine, 3 = Fire right engine
    action = env.action_space.sample()
    
    # Apply the action to the environment
    observation, reward, terminated, truncated, info = env.step(action)
    """
    print("observation:", observation)
    print("info:", info)
    print()
    print("observation_space:", env.observation_space)
    print("action_space:     ", env.action_space)"""

    # If the episode finishes (lander crashes or lands safely), reset it
    if terminated or truncated:
        observation, info = env.reset()

# 4. Close the environment when done
env.close()

env.action_space.sample() — the value comes from a dice roll
action = env.action_space.sample()
self.np_random.integers(0, 4)   # uniform draw from {0,1,2,3}


heuristic_controller(observation) — the value is computed from the state
action = heuristic_controller(observation)

Every single call is a deterministic function of the 8 numbers in observation. Watch t=0 through t=5: x is drifting negative (lander sliding left of the pad, e.g. x=-0.004 → -0.022), and vx is consistently around -0.35 to -0.4 (moving left fast). The controller:

* Computes angle_target = clip(x*0.5 + vx*1.0, -0.4, 0.4) — because vx is strongly negative, this pins to -0.4 (max allowed left lean) every single step here.

* Computes angle_error = (angle_target - angle)*0.5 - vang*1.0 — comes out around -0.29 to -0.07, consistently negative.

* Since angle_error < -0.05, it fires action 3 (right thruster) — every single one of these 6 steps, not by chance, but because the physics genuinely calls for it: to tilt left (toward -0.4), you fire the right thruster to rotate the ship counter-clockwise.

In [ ]:
# If needed (first time only):
# pip install "gymnasium[box2d]"
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random



# 1. Create the environment
# render_mode="human" tells Gymnasium to open a window and show the visual
env = gym.make("LunarLander-v3", render_mode="human")

# 2. Reset the environment to start a new episode
observation, info = env.reset(seed=42)


def heuristic_controller(obs):
    x, y, vx, vy, angle, vang, leg1, leg2 = obs

    # Desired angle: lean toward the pad, but not too much
    angle_target = np.clip(x * 0.5 + vx * 1.0, -0.4, 0.4)
    # Desired hover height: come down more gently the further off-center we are
    hover_target = 0.55 * abs(x)

    angle_error = (angle_target - angle) * 0.5 - vang * 1.0
    hover_error = (hover_target - y) * 0.5 - vy * 0.5

    if leg1 or leg2:
        # already touching -- stop trying to rotate, just cushion the descent
        angle_error = 0
        hover_error = -vy * 0.5

    if hover_error > abs(angle_error) and hover_error > 0.05:
        return 2       # main engine: slow the descent
    elif angle_error < -0.05:
        return 3       # right thruster: rotate one way
    elif angle_error > 0.05:
        return 1        # left thruster: rotate the other way
    else:
        return 0        # do nothing

# 3. Run a loop for 1000 frames
for _ in range(1000):
    # Select a random action from the action space
    action = heuristic_controller(observation)
    
    # Apply the action to the environment
    observation, reward, terminated, truncated, info = env.step(action)
    """
    print("observation:", observation)
    print("info:", info)
    print()
    print("observation_space:", env.observation_space)
    print("action_space:     ", env.action_space)"""

    # If the episode finishes (lander crashes or lands safely), reset it
    if terminated or truncated:
        observation, info = env.reset()

# 4. Close the environment when done
env.close()

"""heuristic_controller(observation) — computes the action from the state. 
It reads x, vx to figure out how much the lander should tilt, compares that to the current angle, and picks whichever thruster corrects the error. Same observation always gives the same action.

In the trace above: the lander was drifting left (x and vx both negative) for all 6 steps.

Random picked 0, 2, 2, 0, 3, 0 — no pattern, ignores the drift.
Heuristic picked 3, 3, 3, 3, 3, 3 — right thruster every time, because that's what rotates the ship to counter a leftward drift, and it kept recomputing the same correct answer as long as the drift persisted."""